In [1]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download "
               "and extraction.")
        return

    with urllib.request.urlopen(url) as response:   
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)              
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


In [2]:
import pandas as pd

df = pd.read_csv(data_file_path, sep='\t', header=None, names=['Label', 'Text'])
print(df.head())

  Label                                               Text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
df['Label'].value_counts()

Label
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
#lets balance the dataset

def create_balnced_dataset(df):
    num_spam = df[df['Label'] == 'spam'].shape[0]
    ham_df = df[df['Label'] == 'ham'].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_df, df[df['Label'] == 'spam']])
    balanced_df = balanced_df.sample(balanced_df.shape[0], random_state=123)
    return balanced_df

balanced_df = create_balnced_dataset(df)
print(balanced_df['Label'].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [5]:
balanced_df['Label'] = balanced_df['Label'].map({"ham":0, "spam":1})

In [6]:
# training : validataion : test= 70: 10: 20
def ramdom_split(df, train_frac, validation_frac):
    df = df.reset_index(drop=True)
    train_end = int(train_frac * len(df))
    validation_end = train_end + int(validation_frac * len(df))
    train_df = df[:train_end]
    val_df = df[train_end: validation_end]
    test_df = df[validation_end:]

    return train_df, val_df, test_df

train_df, val_df, test_df = ramdom_split(balanced_df, 0.7, 0.1)

In [7]:
train_df.to_csv("train.csv", index=None)
val_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [8]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})

[50256]

In [9]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [tokenizer.encode(text) for text in self.data['Text']]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]
        
        self.encoded_texts = [
            encoded_text + [pad_token] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        enc_text = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (torch.tensor(enc_text, dtype=torch.long),
                torch.tensor(label, dtype=torch.long))

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            if len(encoded_text) > max_length:
                max_length = len(encoded_text) 
                
        return max_length

In [10]:
train_dataset = SpamDataset(csv_file='train.csv', tokenizer=tokenizer)
val_dataset = SpamDataset(csv_file='validation.csv', tokenizer=tokenizer, max_length=train_dataset.max_length)
test_dataset = SpamDataset(csv_file='test.csv', tokenizer=tokenizer, max_length=train_dataset.max_length)

In [11]:
train_dataset.max_length

120

In [12]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)

In [13]:
for input_batch, target_batch in train_loader:
    pass
print(input_batch.shape, target_batch.shape)

torch.Size([8, 120]) torch.Size([8])


In [14]:
print(len(train_loader), "training batches")
print(len(val_loader), "validation batches")
print(len(test_loader), "test batches")

130 training batches
19 validation batches
38 test batches


In [15]:
# lets initialize the model with pretrained weights
model_name = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size": 50257,         
    "context_length": 1024,      
    "drop_rate": 0.0,            
    "qkv_bias": True             
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}
}

In [16]:
BASE_CONFIG.update(model_configs[model_name])

In [17]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent.parent.parent
sys.path.insert(0, str(project_root))

from gpt_for_text_generation.src.gpt import GPTModel
from pretraining.src.gpt_generate import load_weights_into_gpt, download_and_load_gpt2

In [18]:
model_size = model_name.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size, "gpt2")

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [19]:
gpt = GPTModel(BASE_CONFIG)
load_weights_into_gpt(gpt, params)
gpt.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [20]:
from gpt_for_text_generation.src.gpt import generate_text_simple
from pretraining.src.pretrain import text_to_token_ids, token_ids_to_text
tokenizer = tiktoken.get_encoding("gpt2")
input_ids = generate_text_simple(gpt, idx=text_to_token_ids(INPUT_PROMPT, tokenizer), max_new_tokens=25, context_size=BASE_CONFIG["context_length"])
print(token_ids_to_text(input_ids, tokenizer))



Every effort moves forward, but it's not enough.

"I'm not going to sit here and say, 'I'm not


In [21]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

input_ids = generate_text_simple(gpt, idx=text_to_token_ids(text_2, tokenizer), max_new_tokens=25, context_size=BASE_CONFIG["context_length"])
print(token_ids_to_text(input_ids, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have


In [22]:
#lets freeze the model
for param in gpt.parameters():
    param.requires_grad = False

In [23]:
import torch
import torch.nn as nn

In [24]:
torch.manual_seed(123)
num_classes = 2
gpt.out_head = nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

In [25]:
print(gpt)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [26]:
for param in gpt.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in gpt.final_norm.parameters():
    param.requires_grad = True

In [27]:
# now model will output 2 logits for each input token
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(inputs)

tensor([[5211,  345,  423,  640]])


In [28]:
logits = gpt(inputs)
print(logits.shape)
print(logits)

torch.Size([1, 4, 2])
tensor([[[-1.5883,  0.9920],
         [-3.7208,  7.4510],
         [-2.2642,  6.6005],
         [-3.5965,  3.9889]]], grad_fn=<ViewBackward0>)


In [29]:
#to finetune the model we will focus on the last ouput token
print("last output token logits: ", logits[:, -1, :])

last output token logits:  tensor([[-3.5965,  3.9889]], grad_fn=<SelectBackward0>)


In [30]:
probas = torch.softmax(logits[:, -1, :], dim=-1)
label = torch.argmax(probas, dim=-1)
print(label.item())

1


In [31]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]
            predicted_labels = torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    
    return correct_predictions/ num_examples

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device)

torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, gpt, device, 10)
val_accuracy = calc_accuracy_loader(val_loader, gpt, device, 10)
test_accuracy = calc_accuracy_loader(test_loader, gpt, device, 10)

In [33]:
print(f"training accuracy {train_accuracy*100:.2f}%")
print(f"validation accuracy {val_accuracy*100:.2f}%")
print(f"test accuracy {test_accuracy*100:.2f}%")

training accuracy 46.25%
validation accuracy 45.00%
test accuracy 48.75%


In [34]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

In [35]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else: 
            break
    
    return total_loss / num_batches

In [36]:
#lets test this
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, gpt, device, 5)
    val_loss = calc_loss_loader(val_loader, gpt, device, 5)
    test_loss = calc_loss_loader(test_loader, gpt, device, 5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")


Training loss: 2.445
Validation loss: 2.575
Test loss: 2.314


In [37]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
    val_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, val_loss

In [38]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    num_examples, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            num_examples += input_batch.shape[0]
            global_step += 1


            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Epoch {epoch + 1} (step {global_step:06d}) "
                      f"Train loss {train_loss:.3f} "
                      f"Val loss {val_loss:.3f}"
                      )
            
        train_accuracy = calc_accuracy_loader(train_loader, model, device, eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, eval_iter)
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)
    return train_losses, val_losses, train_accs, val_accs, num_examples


In [39]:
# torch.manual_seed(123)
# optimizer = torch.optim.AdamW(gpt.parameters(), lr=5e-5, weight_decay=0.1)
# num_epochs = 5
# train_classifier_simple(gpt, train_loader, val_loader, optimizer, device, num_epochs, eval_freq=50, eval_iter=5)


In [40]:
# after training the model in the kaggle, i will load the modified weights and test them

In [41]:
classification = torch.load('model_and_optimizer_classification.pth', map_location=torch.device('cpu'))
gpt.load_state_dict(classification['model_state_dict'])
optimizer = torch.optim.AdamW(gpt.parameters(), lr=5e-5, weight_decay=0.1)
optimizer.load_state_dict(classification['optimizer_state_dict'])
gpt.train()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [42]:
train_accuracy = calc_accuracy_loader(train_loader, gpt, device)
val_accuracy = calc_accuracy_loader(val_loader, gpt, device)
test_accuracy = calc_accuracy_loader(test_loader, gpt, device)
print(f"training accuracy {train_accuracy*100:.2f}%")
print(f"validation accuracy {val_accuracy*100:.2f}%")
print(f"test accuracy {test_accuracy*100:.2f}%")

training accuracy 97.12%
validation accuracy 97.32%
test accuracy 95.67%


In [43]:
# now lets test the model on some examples.
# for this lets first create a funciton which will take a text input and after processing it will return the predicted label
def classify_review(text, model, device, tokenizer, max_length=None, pad_token_id = 50256):
    model.eval()
    token_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    token_ids = token_ids[:min(max_length, supported_context_length)]
    token_ids += [pad_token_id] * (max_length - len(token_ids))
    token_ids = torch.tensor(token_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        logits = gpt(token_ids)[:, -1, :]
    label = torch.argmax(logits, dim=-1).item()
    return "spam" if label == 1 else "not spam"

In [44]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)
print(classify_review(text_1, gpt, device, tokenizer, max_length=train_dataset.max_length))

spam


In [50]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(classify_review(text_2, gpt, device, tokenizer, max_length=train_dataset.max_length))

not spam
